[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/optics/black_sandwich/black_sandwich.ipynb)

# Black Polarization Sandwiches

This example computes the electric field and optical operator of light passing sequentially through two identical stacks of crossed polarizers containing a rotating half-wave plate, known as black polarization sandwiches after [Berry and Dennis (2004)](https://michaelberryphysics.wordpress.com/wp-content/uploads/2013/07/berry361.pdf). A single wave plate inserted between crossed polarizers restores transmission whenever its fast axis turns oblique to both polarizers, yet placing a second identical sandwich immediately behind the first blocks all light at every plate angle because the first stack's exit polarizer lies perpendicular to the second stack's entrance polarizer.

In spacetime geometric algebra, the electric field is modelled as an extrinsic bivector and each polarizer or fast axis is modelled as an intrinsic spatial antivector. Pairing the field with the worldsheet `axis & time` reads the transmitted amplitude, and multiplying by the dual field `(axis & time).dual()` builds the rank-one polarizer operator `polarizer(axis, axis)` and the half-wave plate `2 * polarizer(plate_axis, plate_axis) - Field`, so one stack is the composition `sandwich = exit_polarizer(plate(entrance))` whose square `sandwich(sandwich)` vanishes identically.

In Jones calculus, choosing transverse basis axes $(x, y)$ writes the crossed polarizers and rotated half-wave plate as the $2 \times 2$ matrices $P_x = \bigl(\begin{smallmatrix}1 & 0\\ 0 & 0\end{smallmatrix}\bigr)$, $P_y = \bigl(\begin{smallmatrix}0 & 0\\ 0 & 1\end{smallmatrix}\bigr)$, and $H(\theta) = \bigl(\begin{smallmatrix}\cos 2\theta & \sin 2\theta\\ \sin 2\theta & -\cos 2\theta\end{smallmatrix}\bigr)$, giving a nilpotent off-diagonal product $S(\theta) = P_y H(\theta) P_x = \bigl(\begin{smallmatrix}0 & 0\\ \sin 2\theta & 0\end{smallmatrix}\bigr)$ with $S(\theta)^2 = 0$.

In [ ]:
# In Colab, install numga and its examples from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
%matplotlib inline
from collections.abc import Iterator
from itertools import accumulate

import numpy as np

from numga import Algebra, NumpyContext, stack
from examples.util.animation import show_states
from examples.optics.black_sandwich import render, scenarios

np.set_printoptions(precision=3, suppress=True)

# Spacetime `x+y+z+t-`, space positive and time negative.
ga = Algebra("x+y+z+t-")
mv = NumpyContext(ga).multivector
Scalar = ga.gatype.scalar()                                                    # Scalar+
Field = ga.gatype.bivector()                                                   # Bivector+
Direction = ga.gatype.antivector()                                             # Antivector-
OpticalMap = ga.gatype((Field, Field))                                         # Field+ <- Field+

# Unit timelike direction of the apparatus rest frame, pairing to +1 with `mv.t`.
time = -mv.t.dual()                                                            # [] Direction-
# Unit electric field polarized along a spatial direction, dual to the spacetime plane swept out in time.
field_along = (Direction & time).dual()                                        # [] Field+ <- Direction-
# Rank-one optical projection onto one axis weighted by the field's pairing with a second axis's worldsheet.
polarizer = field_along * ((Direction & time) & Field)                         # [] Field+ <- (Direction-, Direction-, Field+)

## Rotating half-wave plates between crossed polarizers

Binding both direction arguments of `polarizer` to a unit axis gives a linear polarizer along that axis, and `2 * polarizer(plate_axis, plate_axis) - Field` reflects the transverse polarization across `plate_axis` as an ideal half-wave plate. Both half-wave plates swing synchronously from horizontal to vertical and back while incident `x`-polarized light enters from the left.

In [ ]:
n_frames = scenarios.n_frames
frame_duration_ms = scenarios.frame_duration_ms
n_half_frames = n_frames // 2 + 1
input_axis = mv.x.dual()                                                       # [] Direction-
output_axis = mv.y.dual()                                                      # [] Direction-

In [ ]:
def stack_stage_fields(entrance: OpticalMap, plate: OpticalMap, exit_polarizer: OpticalMap,
                       incident_field: Field) -> Field:
    """Propagating `incident_field` (`[...] Field+`) sequentially through two identical
    `(entrance, plate, exit_polarizer)` stacks collects the electric field before and after every
    component (`[..., 7] Field+`)."""
    components = (entrance, plate, exit_polarizer) * 2
    stage_fields = tuple(accumulate(components, lambda field, component: component(field), initial=incident_field))
    batch_shape = stage_fields[-1].shape
    return stack([field.broadcast_to(batch_shape) for field in stage_fields], axis=-1)  # [..., 7] Field+


def plate_swing_states(n_half_frames: int) -> Iterator[tuple[Field, Direction, OpticalMap]]:
    """Yields `(stage_fields, plate_axis, sandwich)` (`[7] Field+`, `[] Direction-`, `[] Field+ <- Field+`)
    across `n_half_frames` angles as both half-wave plates swing smoothly from `0` to `90` degrees."""
    cycle = np.linspace(0, np.pi, n_half_frames)
    plate_angles = (1 - np.cos(cycle)) * np.pi / 4
    entrance = polarizer(input_axis, input_axis)                               # [] Field+ <- Field+
    exit_polarizer = polarizer(output_axis, output_axis)                       # [] Field+ <- Field+
    incident_field = field_along(input_axis)                                   # [] Field+

    for plate_angle in plate_angles:
        plate_rotor = (mv.xy * (-plate_angle / 2)).exp()                       # [] Rotor+
        plate_axis = plate_rotor >> input_axis                                 # [] Direction-
        # A half-wave plate preserves the fast-axis component and reverses the orthogonal transverse component.
        plate = 2 * polarizer(plate_axis, plate_axis) - Field                  # [] Field+ <- Field+
        sandwich = exit_polarizer(plate(entrance))                             # [] Field+ <- Field+
        stage_fields = stack_stage_fields(entrance, plate, exit_polarizer, incident_field)  # [7] Field+
        yield stage_fields, plate_axis, sandwich


recorded_states = show_states(
    plate_swing_states(n_half_frames), render.sandwich_animation, input_axis, output_axis,
    name="black_sandwich", duration_ms=frame_duration_ms, n_frames=n_frames,
)

Light travels from left to right through two identical three-element stacks, where dark strokes mark the fixed crossed polarizers, orange strokes mark the rotating half-wave fast axes, blue segments along the axis scale in opacity with beam intensity, and the seven overhead rings show the polarization direction before and after every optical element.

## Nilpotence of the sandwich operator

At a 45-degree fast-axis angle, one stack transmits the entire incident amplitude turned onto the vertical exit axis, yet its image lies completely inside the kernel of the identical second stack so `single_stack(single_stack)` vanishes identically.

In [ ]:
fields = stack([state[0] for state in recorded_states])                        # [n_half_frames, 7] Field+
sandwiches = stack([state[2] for state in recorded_states])                    # [n_half_frames] Field+ <- Field+
brightest_index = n_frames // 4
single_stack = sandwiches[brightest_index]                                     # [] Field+ <- Field+
double_stack = single_stack(single_stack)                                      # [] Field+ <- Field+

print("One stack at 45 degrees:")
print(single_stack.coefficient_table())
print("Two stacks in series:")
print(double_stack.coefficient_table())

In [ ]:
# checks
transmitted = fields[:, 3] & (output_axis & time)                              # [n_half_frames] Scalar+
assert (transmitted.max() - 1).abs() < 1e-12
assert (np.abs(sandwiches(sandwiches).kernel) < 1e-12).all()
assert (np.abs(fields[:, 4:].kernel) < 1e-12).all()